# AML Data Preprocessing

Replicates the **CSV-level column transformations** from `muleaccount/Multi-GNN/format_kaggle_files.py`.

### What this notebook does:
1. Loads the raw AML transaction CSV
2. **Parses `Timestamp`** → relative seconds from the start of the dataset
3. **Label-encodes** `Payment Format`, `Receiving Currency`, and `Payment Currency` as integers (shared dict for currencies, same as original)
4. **Merges `From Bank` + `Account` / `To Bank` + `Account`** → unique integer node IDs (`from_id`, `to_id`) via a shared account dict
5. **Adds `EdgeID`** (sequential row index)
6. **Renames / drops** raw columns to match the formatted output schema
7. **Sorts** by `Timestamp` (matching the original `sort(3)`)
8. Saves the result as `data/formatted_transactions.csv`

## 1. Load the raw dataset

In [1]:
import os
import sys
import pandas as pd
from datetime import datetime

sys.path.insert(0, os.path.dirname(os.path.abspath('get_dataset.py')))
from get_dataset import load_aml_dataset

df = load_aml_dataset()
print(f"Loaded {len(df)} rows")
print(f"Columns : {df.columns.tolist()}")

Loaded 15000000 rows
Columns : ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


## 2. Parse `Timestamp` → Relative Seconds

`format_kaggle_files.py` logic:
```python
startTime = datetime(year, month, day)   # midnight of first transaction day
firstTs   = startTime.timestamp() - 10
ts        = datetime_object.timestamp() - firstTs
```

In [ ]:
# Parse raw string timestamps
dt_series = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M')

# firstTs = midnight of the earliest transaction day minus 10 seconds
first_dt   = dt_series.min()
start_time = datetime(first_dt.year, first_dt.month, first_dt.day)
firstTs    = start_time.timestamp() - 10

# Convert to relative integer seconds
df['Timestamp'] = (dt_series.astype('int64') // 10**9 - firstTs).astype('int64')

print(f"Reference epoch (firstTs): {firstTs}")
print(f"Timestamp range: {df['Timestamp'].min()} – {df['Timestamp'].max()} seconds")
df[['Timestamp']].head(3)

## 3. Label-Encode Categorical Columns

Original uses a **single shared `currency` dict** for both `Receiving Currency` and `Payment Currency`, and a separate `paymentFormat` dict — so the same currency string always maps to the same integer across both columns.

In [ ]:
def encode_columns_shared(df, cols):
    """Encode multiple columns with ONE shared label dict (mirrors the original currency dict)."""
    shared = {}
    result = {}
    for col in cols:
        encoded = []
        for val in df[col]:
            if val not in shared:
                shared[val] = len(shared)
            encoded.append(shared[val])
        result[col] = encoded
    return result, shared

def encode_column(series):
    """Encode a single column with its own label dict."""
    label_dict = {}
    encoded = []
    for val in series:
        if val not in label_dict:
            label_dict[val] = len(label_dict)
        encoded.append(label_dict[val])
    return encoded, label_dict

# Currency: shared dict across both currency columns
currency_encoded, currency_dict = encode_columns_shared(df, ['Receiving Currency', 'Payment Currency'])
df['Receiving Currency'] = currency_encoded['Receiving Currency']
df['Payment Currency']   = currency_encoded['Payment Currency']

# Payment Format: separate dict
fmt_encoded, fmt_dict = encode_column(df['Payment Format'])
df['Payment Format'] = fmt_encoded

print("Currency encoding :", currency_dict)
print("Payment Format enc:", fmt_dict)
df[['Receiving Currency', 'Payment Currency', 'Payment Format']].head(3)

## 4. Build `from_id` and `to_id` Node IDs

Original concatenates `From Bank + Account[col 2]` and `To Bank + Account[col 4]` as strings, assigns IDs from a **single shared `account` dict** (so the same account always gets the same integer regardless of whether it's sender or receiver).

In [ ]:
account_dict = {}

def get_account_id(bank, acc):
    key = str(bank) + str(acc)
    if key not in account_dict:
        account_dict[key] = len(account_dict)
    return account_dict[key]

# Detect the second Account column (pandas auto-renames duplicate cols to 'Account.1')
to_acc_col = 'Account.1' if 'Account.1' in df.columns else df.columns[4]

df['from_id'] = [get_account_id(b, a) for b, a in zip(df['From Bank'], df['Account'])]
df['to_id']   = [get_account_id(b, a) for b, a in zip(df['To Bank'],   df[to_acc_col])]

print(f"Total unique accounts (nodes): {len(account_dict):,}")
df[['From Bank', 'Account', 'from_id', 'To Bank', to_acc_col, 'to_id']].head(3)

## 5. Add `EdgeID`, Rename & Drop Columns

Final schema (from the `header` string in `format_kaggle_files.py`):
```
EdgeID, from_id, to_id, Timestamp, Amount Sent, Sent Currency, Amount Received, Received Currency, Payment Format, Is Laundering
```

In [ ]:
# Add EdgeID as sequential 0-based index (matches loop index i in the original)
df.insert(0, 'EdgeID', range(len(df)))

# Rename columns to match the formatted schema
df.rename(columns={
    'Amount Paid'       : 'Amount Sent',
    'Payment Currency'  : 'Sent Currency',
    'Receiving Currency': 'Received Currency',
}, inplace=True)

# Drop raw bank/account string columns — already encoded into from_id / to_id
cols_to_drop = ['From Bank', 'Account', 'To Bank', to_acc_col]
df.drop(columns=cols_to_drop, inplace=True)

# Reorder to the exact formatted output schema
df = df[[
    'EdgeID', 'from_id', 'to_id', 'Timestamp',
    'Amount Sent', 'Sent Currency',
    'Amount Received', 'Received Currency',
    'Payment Format', 'Is Laundering'
]]

print("Columns:", df.columns.tolist())
df.head(3)

## 6. Sort by Timestamp

Mirrors `formatted[:,:,sort(3)]` in the original (sort by column index 3 = `Timestamp`).

In [ ]:
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Shape             : {df.shape}")
print(f"Timestamp range   : {df['Timestamp'].min()} – {df['Timestamp'].max()} seconds")
print(f"Illicit ratio     : {df['Is Laundering'].sum():,} / {len(df):,} = {df['Is Laundering'].mean()*100:.3f}%")
df.head(5)

## 7. Save Formatted Transactions

In [ ]:
out_dir  = os.path.join(os.path.dirname(os.path.abspath('.')), 'model', 'data')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'formatted_transactions.csv')

df.to_csv(out_path, index=False)
print(f"Saved  → {out_path}")
print(f"Size   : {os.path.getsize(out_path) / 1e6:.1f} MB")